# Mesh 1 · A mesh from scratch

**Fundamentals, part 1 of 4.** Companion to [Week 05 §1–3 and §8](../README.md).

A facade, a shell, a terrain, a 3D print: each is one connected surface,
and most questions about it are about *connections*. What is next to this
panel? Where is the edge? This notebook builds the data structure that can
answer them, one call at a time.

| You will | Section |
| --- | --- |
| see why a list of polygons is not enough | 1 |
| add vertices and faces by hand | 2–3 |
| get a face the right way up | 4 |
| build a grid with a dictionary | 5 |
| use the ready-made constructors | 6 |
| catch a broken mesh with one integer | 7 |

Run the cells in order. Before each one, **say what you expect**, then run.

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in its lecture folder or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_plan, draw_points, ramp, row, view3d

from compas.datastructures import Mesh

## 1 · Why not just a list of polygons?

![A triangle soup stores shared corners twice; a mesh stores them once and refers to them by key](../figures/mesh-soup-vs-mesh.svg)

Two panels side by side, written the obvious way: each as its own list of
corner coordinates.

In [ ]:
soup = [
    [(0, 0, 0), (1, 0, 0), (1, 1, 0), (0, 1, 0)],
    [(1, 0, 0), (2, 0, 0), (2, 1, 0), (1, 1, 0)],
]
corners = [point for polygon in soup for point in polygon]
print("corner points stored:   ", len(corners))
print("different corner points:", len(set(corners)))

Eight stored, six real. The two panels touch along an edge, and **nothing in
the data says so**. To find out you would compare coordinates, and with
real numbers (0.1 + 0.2 is not 0.3) that comparison is not even safe.

A mesh stores each point **once** and lets faces *refer* to it. Shared
reference, shared edge: adjacency becomes a lookup.

## 2 · Vertices are keys

`add_vertex` stores a point and hands back an integer, its **key**. Keep
the key: everything else is built from keys, never from coordinates.

In [ ]:
mesh = Mesh()
a = mesh.add_vertex(x=0, y=0, z=0)
b = mesh.add_vertex(x=1, y=0, z=0)
c = mesh.add_vertex(x=1, y=1, z=0)
d = mesh.add_vertex(x=0, y=1, z=0)
print("keys:", a, b, c, d)
print("vertex", c, "is at", mesh.vertex_coordinates(c))
print("vertices:", mesh.number_of_vertices(), "| faces:", mesh.number_of_faces())

Four points and no surface yet. A point cloud, so far.

## 3 · A face is a cycle of keys

![A 3 by 2 grid mesh with vertex keys, face keys and three highlighted edges](../figures/mesh-anatomy.svg)

A face is an ordered list of vertex keys, closed back on itself. It also
gets a key of its own.

In [ ]:
f = mesh.add_face([a, b, c, d])
print("face key:", f)
print("its vertices:", mesh.face_vertices(f))
print("edges appeared by themselves:", list(mesh.edges()))
draw_mesh(mesh, "one face, four vertices", face_labels=True, size=(420, 340))

You never called `add_edge`. **An edge exists because a face uses it**:
that is why a mesh cannot have a loose edge, and a graph (Week 06) can.

## 4 · The order of the cycle is the direction it faces

![The same square wound counter-clockwise has normal +Z; wound clockwise, -Z](../figures/mesh-winding.svg)

Counter-clockwise seen from above: the normal points up. The same four
keys the other way round: it points down. Nothing else differs.

In [ ]:
up = Mesh()
keys = [up.add_vertex(x=x, y=y, z=0) for x, y in [(0, 0), (1, 0), (1, 1), (0, 1)]]
up.add_face(keys)

down = Mesh()
keys = [down.add_vertex(x=x, y=y, z=0) for x, y in [(0, 0), (1, 0), (1, 1), (0, 1)]]
down.add_face(list(reversed(keys)))

print("counter-clockwise:", up.face_normal(0))
print("clockwise:        ", down.face_normal(0))

The two meshes draw identically in plan. In a renderer one is lit and one
is black; in a 3D print one is inside-out; in a solar study one faces the
ground. **Check the normal, not the picture.**

## 5 · A grid, with a dictionary doing the bookkeeping

Typing every face by hand does not scale. The pattern that does: remember
which key sits at grid position `(i, j)`, then look the corners up. This is
checkpoint 1.

In [ ]:
def grid(nx, ny, spacing=1.0):
    """An nx by ny grid of quads. Keys run along x first; every face is counter-clockwise."""
    mesh = Mesh()
    key = {}
    for j in range(ny + 1):
        for i in range(nx + 1):
            key[i, j] = mesh.add_vertex(x=i * spacing, y=j * spacing, z=0)
    for j in range(ny):
        for i in range(nx):
            mesh.add_face([key[i, j], key[i + 1, j], key[i + 1, j + 1], key[i, j + 1]])
    return mesh


g = grid(4, 3)
print("V E F =", g.number_of_vertices(), g.number_of_edges(), g.number_of_faces())
print("face 0 uses", g.face_vertices(0), "and face 1 uses", g.face_vertices(1))
draw_mesh(g, "grid(4, 3)", face_labels=True)

Faces 0 and 1 both contain keys 1 and 6. That shared pair **is** the edge
between them. Compare with section 1: same picture, and now the data knows.

Count check, before trusting it: a 4 × 3 grid should have 5 × 4 = 20
vertices and 4 × 3 = 12 faces. Edges: 4 × 4 across plus 5 × 3 up = 31.

## 6 · Ready-made constructors

You will rarely build a mesh vertex by vertex. These are the usual ways in.

In [ ]:
import compas
from compas.geometry import Box, Sphere

sources = {
    "from_meshgrid(dx=4, nx=4)": Mesh.from_meshgrid(dx=4, nx=4),
    "from_polyhedron(6), a cube": Mesh.from_polyhedron(6),
    "from_shape(Box(2, 1, 1))": Mesh.from_shape(Box(2, 1, 1)),
    "from_shape(Sphere(1), u=16, v=8)": Mesh.from_shape(Sphere(1.0), u=16, v=8),
    "from_vertices_and_faces(...)": Mesh.from_vertices_and_faces(
        [[0, 0, 0], [1, 0, 0], [1, 1, 0], [0, 1, 0]], [[0, 1, 2, 3]]),
    "from_obj(a file)": Mesh.from_obj(compas.get("hypar.obj")),
}
for name, m in sources.items():
    print(f"{name:34s} V {m.number_of_vertices():4d}  E {m.number_of_edges():4d}  F {m.number_of_faces():4d}")

In [ ]:
view3d((sources["from_obj(a file)"], {"show_edges": True}))

`from_vertices_and_faces` is the one to remember: a list of points and a
list of index lists is how every other program (Rhino, Blender, a scanner)
hands a mesh over.

## 7 · Euler: one integer that catches a broken mesh

![A flat grid gives 1, a grid with a hole gives 0, two quads with duplicated vertices give 2](../figures/mesh-euler.svg)

`V − E + F` depends only on the *kind* of surface: 1 for a sheet, 2 for a
closed ball, 0 for a sheet with one hole. Not on the size, not on the
shape. So it is a free test.

In [ ]:
for nx, ny in [(1, 1), (4, 3), (20, 10)]:
    m = grid(nx, ny)
    print(f"grid({nx}, {ny}):  {m.number_of_vertices()} - {m.number_of_edges()} + {m.number_of_faces()} = {m.euler()}")
print("a cube:", Mesh.from_polyhedron(6).euler())

Now the soup from section 1, turned into a mesh **naively**: every polygon
brings its own four vertices.

In [ ]:
bad = Mesh()
for polygon in soup:
    bad.add_face([bad.add_vertex(x=x, y=y, z=z) for x, y, z in polygon])

good = grid(2, 1)
print("naive:   V E F =", bad.number_of_vertices(), bad.number_of_edges(), bad.number_of_faces(), "-> euler", bad.euler())
print("grid:    V E F =", good.number_of_vertices(), good.number_of_edges(), good.number_of_faces(), "-> euler", good.euler())
row(draw_mesh(good, "shared vertices: euler 1", size=(400, 260)),
    draw_mesh(bad, "duplicated vertices: euler 2", size=(400, 260)))

The pictures are the same. The integers are not: euler 2 means **two
separate sheets**, lying next to each other. Ask the bad mesh for the
neighbour of its first face and it has none.

The repair is `weld`: vertices closer than a tolerance become one.

In [ ]:
print("face 0's neighbours, before:", bad.face_neighbors(0))
bad.weld()
print("after weld: V E F =", bad.number_of_vertices(), bad.number_of_edges(), bad.number_of_faces(), "-> euler", bad.euler())
print("face 0's neighbours, after: ", bad.face_neighbors(0))

That was the whole argument of section 1, run backwards: the soup became a
mesh the moment its vertices were shared.

## What to keep

- A vertex is a **key**; a face is a **cycle of keys**; edges follow.
- The **order** of a face's keys is its normal.
- Two faces are neighbours because they **share keys**, not because their
  coordinates agree.
- `mesh.euler()` after you build anything. 1 for a sheet, 2 for a solid.

Next: [`02_walking_a_mesh`](02_walking_a_mesh.ipynb), where the mesh starts
answering questions.

## Your turn
1. Build `grid(3, 3)` and add one more face **clockwise** on top of an
   existing one. What does COMPAS say, and why can a mesh not hold it?
2. Predict V, E, F and the Euler characteristic of `Mesh.from_polyhedron(20)`
   before you print them.
3. A triangulated scan arrives as 3000 separate triangles. What will
   `euler()` say before welding? And after, if the scan is one sheet?
4. Write `strip(n)`: a row of `n` triangles sharing vertices, using a
   dictionary as `grid` does.